# run_baseline_comparison_gate

该 Notebook 是 `baseline_comparison_gate` 的 Colab 执行入口。它只负责 Colab 会话编排, 正式逻辑全部委托给仓库脚本和 `experiments/baseline_comparison_gate/` 模块。

当前 Notebook 的目标是完成阶段三正式 runner 前置准备: 检查并解压阶段二 real-video VAE 正式结果包、拉取外部 baseline 上游源码、执行真实 smoke、汇总 real smoke、冻结 formal input contract, 并生成正式 scoring work-item 计划。当前 work-item 计划不包含外部 baseline 分数, 不能支持论文 claim。


## 0. 使用边界

- 本 Notebook 不直接写出正式 `records`、`thresholds`、`tables`、`figures` 或 `reports` 逻辑。
- 本 Notebook 不把 `external_baselines/` 提交进仓库。
- `external_baselines/` 是冷启动缓存目录, 每次 Colab 运行时通过 `scripts/prepare_baselines/fetch_external_baselines.py` 重新拉取。
- 真实 baseline adapter、权重 digest 和正式比较表仍需要后续阶段继续实现。

In [ ]:
# 可修改配置区
REPO_URL = "https://github.com/RICHAAARC/TSTW-VW.git"
REPO_DIR = "/content/TSTW-VW"
GIT_REF = "explicit-sync"
DRIVE_MOUNT = "/content/drive"
DRIVE_RESULT_ROOT = "/content/drive/MyDrive/TSTW/results"
BASELINE_CONFIG_DIR = f"{REPO_DIR}/configs/baselines"
EXTERNAL_BASELINES_ROOT = f"{REPO_DIR}/external_baselines"
SESSION_RUN_ROOT = "/content/TSTW_runtime/runs/baseline_comparison_smoke"
REAL_VIDEO_VAE_PACKAGE_ROOT = "/content/drive/MyDrive/TSTW/results/real_video_vae_latent_probe/shard_aggregated/REPLACE_WITH_COMPLETED_STAGE_TWO_AGGREGATED_RUN"
AUTO_RESOLVE_REAL_VIDEO_VAE_PACKAGE_ROOT = True  # 当路径仍是占位符时, 自动选择 real_video_vae_latent_probe/shard_aggregated 下最新的阶段二聚合结果包。
LOCAL_REAL_VIDEO_VAE_EXTRACT_ROOT = "/content/TSTW_runtime/input_packages"
REAL_VIDEO_VAE_INPUT_CHECK_PATH = "/content/TSTW_runtime/input_checks/real_video_vae_for_baseline_check.json"
EXTRACT_REAL_VIDEO_VAE_PACKAGE = True  # 是否检查并解压阶段二结果包。
RUN_VIDEOSEAL_REAL_SMOKE = False  # 是否运行 external_videoseal 的真实 smoke。已存在可用 smoke summary 时建议保持 False, 避免重复生成 Drive 结果包。
VIDEOSEAL_INPUT_VIDEO_PATH = ""  # external_videoseal smoke 的可选输入视频路径。留空表示脚本自动生成确定性测试视频; 填入路径则使用该真实视频做 smoke。
RUN_RIVAGAN_REAL_SMOKE = False  # 是否运行 external_rivagan 的真实 smoke。已存在可用 smoke summary 时建议保持 False, 避免重复生成 Drive 结果包。
RIVAGAN_INPUT_VIDEO_PATH = ""  # external_rivagan smoke 的可选输入视频路径。留空表示脚本自动生成确定性测试视频; 填入路径则使用该真实视频做 smoke。
RUN_BASELINE_COMPARISON_SMOKE = True  # 是否运行轻量 comparison smoke。该 smoke 会按 baseline 分别落盘到 <baseline>/comparison_smoke。
RUN_HIDDEN_FRAMEWISE_REAL_SMOKE = False  # 是否运行 external_hidden_framewise 的真实 smoke。已存在可用 smoke summary 时建议保持 False, 避免重复生成 Drive 结果包。
HIDDEN_FRAMEWISE_INPUT_VIDEO_PATH = ""  # external_hidden_framewise smoke 的可选输入视频路径。留空表示脚本自动生成确定性测试视频; 填入路径则使用该真实视频做 smoke。
RUN_BASELINE_REAL_SMOKE_SUMMARY = False  # 是否重新汇总三个 real smoke 结果。已有 baseline_real_smoke_summary_latest 时建议保持 False。
BASELINE_REAL_SMOKE_SUMMARY_DIR = "/content/drive/MyDrive/TSTW/results/baseline_comparison_gate/baseline_real_smoke_summary_latest"
BASELINE_FORMAL_RUN_ROOT = "/content/TSTW_runtime/runs/baseline_comparison_formal"
EXTRACTED_REAL_VIDEO_VAE_PACKAGE_ROOT = ""  # 阶段二包解压后的本地根目录, 留空表示由检查脚本根据 zip 根目录自动回填。
BASELINE_STAGE_TWO_ARTIFACT_ROOTS = []  # 可选: 阶段二 artifact fallback 根目录列表。聚合瘦身包不含 source video 时, 可填 shard_runs/<family>/compat_run_root。
AUTO_RESOLVE_BASELINE_STAGE_TWO_ARTIFACT_ROOTS = True  # 是否从 Drive 的 real_video_vae_latent_probe/shard_runs 自动搜索 compat_run_root 作为 source video fallback。
RUN_BASELINE_FORMAL_SCORING_PLAN = True  # 是否生成正式 scoring work-item 计划。该步骤只生成任务清单, 不执行外部 baseline 检测。
BASELINE_FORMAL_SCORING_PLAN_RUN_ROOT = "/content/TSTW_runtime/runs/baseline_comparison_formal_scoring_plan"
BASELINE_SCORING_SHARD_COUNT = 1  # 外层 shard 总数。
BASELINE_SCORING_SHARD_INDEX = 0  # 当前运行的 shard 索引。
BASELINE_SCORING_BASELINE_NAMES = ["external_hidden_framewise"]  # scoring plan 阶段的 baseline 过滤器。正式运行建议与 BASELINE_FORMAL_SCORING_EXECUTION_BASELINE_NAMES 保持一致, 使 plan 落盘到对应 baseline/scoring_plans。空列表表示三个 baseline 全部进入 multi_baseline 计划。
RUN_BASELINE_FORMAL_SCORING_SMALL_VALIDATION = True  # 是否运行当前 baseline 当前 shard 的 formal scoring execution。
BASELINE_FORMAL_SCORING_EXECUTION_RUN_ROOT = "/content/TSTW_runtime/runs/baseline_comparison_formal_scoring_execution"
BASELINE_FORMAL_SCORING_EXECUTION_BASELINE_NAMES = ["external_hidden_framewise"]  # execution 阶段实际运行的 baseline 列表。论文级 1% FPR 运行建议三个 baseline 分开执行。
BASELINE_FORMAL_SCORING_EXECUTION_MAX_WORK_ITEMS = None  # execution 最多执行的 work item 数。None 表示不截断, 直接运行当前 shard 的全部 work items。
BASELINE_FORMAL_SCORING_WORKER_COUNT = 50  # 三层策略第3层: 当前 shard 内并发 worker 数。当前为 external_hidden_framewise 的激进高并发验证配置。
BASELINE_FORMAL_SCORING_BATCH_SIZE = 1  # 三层策略第3层: 每个 worker 一次领取的 work item 数。
BASELINE_GPU_PROFILE_INTERVAL_SECONDS = 0.5  # GPU profiling 采样间隔秒数。数值越小采样越密, profiling 文件越大。
BASELINE_FORMAL_SCORING_OVERWRITE_DRIVE_RESULT = True  # 当前 shard 结果目录已存在时是否覆盖, Colab 重跑同一 shard 时建议 True。


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import zipfile


def run_command(command, cwd=None):
    """运行命令并在失败时显示 stdout 和 stderr。"""
    print("$", " ".join(command), flush=True)
    completed = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr, flush=True)
    if completed.returncode != 0:
        raise RuntimeError(f"命令失败: {' '.join(command)}")


def run_python_script(script, *args):
    """使用当前 Python 解释器运行仓库脚本。"""
    run_command([sys.executable, script, *args], cwd=REPO_DIR)


def load_stage_two_aggregate_runtime_config():
    """从阶段二聚合包读取 shard 来源元数据, 用于精确定位 source video fallback。"""
    package_root = Path(REAL_VIDEO_VAE_PACKAGE_ROOT)
    check_path = package_root / "stage_two_package_for_baseline_check.json"
    archive_path = None
    if check_path.exists():
        check_payload = json.loads(check_path.read_text(encoding="utf-8"))
        archive_path = Path(check_payload.get("archive_path") or "")
    if archive_path is None or not archive_path.exists():
        packages_root = package_root / "packages"
        zip_candidates = sorted(packages_root.glob("*.zip"), key=lambda path: path.stat().st_mtime, reverse=True) if packages_root.exists() else []
        archive_path = zip_candidates[0] if zip_candidates else None
    if archive_path is None or not archive_path.exists():
        return {}
    with zipfile.ZipFile(archive_path) as archive:
        members = archive.namelist()
        runtime_config_members = [name for name in members if name.endswith("/artifacts/runtime_config.json")]
        if not runtime_config_members:
            return {}
        return json.loads(archive.read(runtime_config_members[0]).decode("utf-8"))


def resolve_stage_two_artifact_roots_from_aggregate_metadata():
    """根据聚合结果元数据精确推导本次需要加载的 shard compat_run_root。"""
    runtime_config = load_stage_two_aggregate_runtime_config()
    source_shard_count = runtime_config.get("source_shard_count") or runtime_config.get("shard_count")
    source_shard_indexes = runtime_config.get("source_shard_indexes")
    source_commit = str(runtime_config.get("git_commit") or "")[:7]
    if not source_shard_count or not source_shard_indexes or not source_commit:
        return []
    shard_runs_root = Path(DRIVE_RESULT_ROOT) / "real_video_vae_latent_probe" / "shard_runs"
    resolved_roots = []
    for shard_index in source_shard_indexes:
        family_name = f"real_video_vae_latent_probe_sc{int(source_shard_count):02d}_si{int(shard_index):02d}_{source_commit}"
        compat_root = shard_runs_root / family_name / "compat_run_root"
        if (compat_root / "artifacts" / "videos" / "source").exists():
            resolved_roots.append(compat_root.as_posix())
        else:
            print({"missing_expected_stage_two_compat_root": compat_root.as_posix()})
    return resolved_roots


def resolve_stage_two_artifact_roots_for_baseline():
    """解析 baseline execution 读取阶段二 source video 所需的 artifact fallback 根目录。"""
    roots = [str(Path(path)) for path in BASELINE_STAGE_TWO_ARTIFACT_ROOTS if str(path).strip()]
    if AUTO_RESOLVE_BASELINE_STAGE_TWO_ARTIFACT_ROOTS:
        metadata_roots = resolve_stage_two_artifact_roots_from_aggregate_metadata()
        if metadata_roots:
            roots.extend(metadata_roots)
        else:
            shard_runs_root = Path(DRIVE_RESULT_ROOT) / "real_video_vae_latent_probe" / "shard_runs"
            if shard_runs_root.exists():
                for compat_root in sorted(shard_runs_root.glob("*/compat_run_root")):
                    if (compat_root / "artifacts" / "videos" / "source").exists():
                        roots.append(compat_root.as_posix())
    deduped = []
    seen = set()
    for root in roots:
        if root not in seen:
            deduped.append(root)
            seen.add(root)
    print({"stage_two_artifact_roots": deduped})
    return deduped


## 1. 挂载 Google Drive

该步骤只负责挂载 Drive。结果目录不会在这里提前创建, 只有 smoke 成功后才会复制完整结果包。

In [ ]:
try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
except Exception as exc:
    print(f"非 Colab 环境或 Drive 挂载失败: {exc}")


## 2. 获取项目仓库

如果 `REPO_URL` 为空, Notebook 假设 `/content/TSTW-VW` 已经存在。否则会从远程仓库冷启动 clone。


In [ ]:
repo_path = Path(REPO_DIR)
if REPO_URL and not repo_path.exists():
    run_command(["git", "clone", REPO_URL, REPO_DIR])
elif not repo_path.exists():
    raise FileNotFoundError("REPO_DIR 不存在, 请设置 REPO_URL 或先上传/克隆仓库。")

os.chdir(REPO_DIR)
if GIT_REF:
    run_command(["git", "fetch", "--all", "--tags"], cwd=REPO_DIR)
    run_command(["git", "checkout", GIT_REF], cwd=REPO_DIR)

print(f"当前仓库目录: {Path.cwd()}")
run_command(["git", "branch", "--show-current"], cwd=REPO_DIR)
run_command(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR)


## 2.1 检查并解压阶段二 real-video VAE 正式结果包

该步骤读取已经落盘到 Google Drive 的阶段二正式 family 结果目录, 检查 LPIPS、CLIP similarity、records、thresholds、tables 和 manifest 是否齐全。检查通过后, 将 zip 兼容包解压到 Colab 会话本地目录, 供后续正式 baseline comparison runner 读取。该步骤不会重跑 real-video VAE。


In [ ]:

if AUTO_RESOLVE_REAL_VIDEO_VAE_PACKAGE_ROOT and "REPLACE_WITH_COMPLETED" in REAL_VIDEO_VAE_PACKAGE_ROOT:
    candidate_parent = Path("/content/drive/MyDrive/TSTW/results/real_video_vae_latent_probe/shard_aggregated")
    if not candidate_parent.exists():
        raise FileNotFoundError(candidate_parent)
    candidate_roots = sorted(
        [path for path in candidate_parent.iterdir() if path.is_dir()],
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not candidate_roots:
        raise FileNotFoundError(f"未找到阶段二 shard 聚合结果包: {candidate_parent}")
    REAL_VIDEO_VAE_PACKAGE_ROOT = str(candidate_roots[0])
    print({"auto_resolved_real_video_vae_package_root": REAL_VIDEO_VAE_PACKAGE_ROOT})

stage_two_args = [
    "--package-root", REAL_VIDEO_VAE_PACKAGE_ROOT,
    "--summary-path", REAL_VIDEO_VAE_INPUT_CHECK_PATH,
]
if EXTRACT_REAL_VIDEO_VAE_PACKAGE:
    stage_two_args.extend(["--extract", "--extract-root", LOCAL_REAL_VIDEO_VAE_EXTRACT_ROOT])
run_python_script("scripts/prepare_baselines/check_real_video_vae_package_for_baseline.py", *stage_two_args)
stage_two_input_summary = json.loads(Path(REAL_VIDEO_VAE_INPUT_CHECK_PATH).read_text(encoding="utf-8"))
if EXTRACT_REAL_VIDEO_VAE_PACKAGE:
    EXTRACTED_REAL_VIDEO_VAE_PACKAGE_ROOT = stage_two_input_summary.get("extracted_package_root") or EXTRACTED_REAL_VIDEO_VAE_PACKAGE_ROOT
if not EXTRACTED_REAL_VIDEO_VAE_PACKAGE_ROOT:
    raise RuntimeError("EXTRACTED_REAL_VIDEO_VAE_PACKAGE_ROOT 为空, 无法进入 baseline comparison。")
print({
    "stage_two_package_ready": stage_two_input_summary.get("package_ready_for_baseline_comparison"),
    "extracted_real_video_vae_package_root": EXTRACTED_REAL_VIDEO_VAE_PACKAGE_ROOT,
})


## 3. 拉取外部 baseline 源码

`external_baselines/` 被 `.gitignore` 排除是合理的: 它属于第三方上游源码缓存, 不能作为本项目提交内容。每次 Colab 冷启动都通过下面的脚本按 manifest 中固定的 commit 重新拉取。

In [ ]:
run_python_script("scripts/prepare_baselines/fetch_external_baselines.py", "--print-plan")


## 3.1 安装 VideoSeal runtime 依赖

该步骤只安装 `external_videoseal` 真实 smoke 或 formal execution 所需依赖。`external_baselines/` 仍然只是运行时缓存目录, 不会进入 git 提交。


In [ ]:
if RUN_VIDEOSEAL_REAL_SMOKE or "external_videoseal" in BASELINE_FORMAL_SCORING_EXECUTION_BASELINE_NAMES:
    run_command([
        sys.executable, "-m", "pip", "install",
        "-r", "external_baselines/external_videoseal/upstream/requirements.txt"
    ], cwd=REPO_DIR)


## 3.2 安装 RivaGAN runtime 依赖

该步骤只安装 `external_rivagan` 真实 smoke 或 formal execution 所需的最小额外依赖。RivaGAN 上游代码硬编码 CUDA 调用, 因此该 smoke 需要 GPU runtime。


In [ ]:
if RUN_RIVAGAN_REAL_SMOKE or "external_rivagan" in BASELINE_FORMAL_SCORING_EXECUTION_BASELINE_NAMES:
    run_command([sys.executable, "-m", "pip", "install", "torch-dct==0.1.5"], cwd=REPO_DIR)


## 3.3 安装 HiDDeN framewise runtime 依赖

该步骤只安装 `external_hidden_framewise` 逐帧真实 smoke 或 formal execution 所需的最小额外依赖。


In [ ]:
if RUN_HIDDEN_FRAMEWISE_REAL_SMOKE or "external_hidden_framewise" in BASELINE_FORMAL_SCORING_EXECUTION_BASELINE_NAMES:
    run_command([sys.executable, "-m", "pip", "install", "opencv-python-headless"], cwd=REPO_DIR)


## 4. Source probe

该步骤检查上游源码树是否包含预期入口线索。它不下载权重, 也不运行真实模型。

In [ ]:
run_python_script("scripts/prepare_baselines/probe_external_baseline_sources.py")


## 5. Colab preflight

该步骤会显示当前真实 baseline 仍然缺少哪些条件, 包括权重 digest 和真实 adapter smoke。

In [ ]:
run_python_script("scripts/prepare_baselines/check_baseline_colab_preflight.py")


## 6. 运行 baseline smoke 并成功后落盘到 Drive

该命令先写入 `/content/TSTW_runtime/runs/`, 只有本地结果完整后才复制到 Drive。comparison smoke 落盘到 `/content/drive/MyDrive/TSTW/results/baseline_comparison_gate/<baseline>/comparison_smoke/<RUN_ID>/`; real smoke 落盘到 `<baseline>/real_smoke/<RUN_ID>/`; formal input contract 落盘到 `<baseline>/formal_inputs/<RUN_ID>/`; scoring plan 落盘到 `<baseline>/scoring_plans/<RUN_ID>/`; scoring execution 落盘到 `<baseline>/shard_runs/<RUN_ID>/`。

In [ ]:
if RUN_BASELINE_COMPARISON_SMOKE:
    comparison_smoke_baselines = BASELINE_FORMAL_SCORING_EXECUTION_BASELINE_NAMES or [
        "external_videoseal",
        "external_rivagan",
        "external_hidden_framewise",
    ]
    for baseline_name in comparison_smoke_baselines:
        run_python_script(
            "scripts/prepare_baselines/run_baseline_comparison_smoke.py",
            "--run-root", f"{SESSION_RUN_ROOT}/{baseline_name}",
            "--result-root", DRIVE_RESULT_ROOT,
            "--baseline-name", baseline_name,
            "--overwrite-result",
        )
else:
    print("已跳过 baseline comparison 轻量 smoke。")


## 8. external_videoseal 真实可运行性 smoke

该步骤会真实加载 VideoSeal、下载权重、计算权重 digest、执行单视频 embed / detect, 并额外执行 H.264 CRF 28 后的 detect。该结果仍然只是单 baseline smoke, 不能替代正式 fixed-FPR baseline comparison。


In [ ]:
if RUN_VIDEOSEAL_REAL_SMOKE:
    videoseal_args = [
        "--run-root", "/content/TSTW_runtime/runs/external_videoseal_real_smoke",
        "--result-root", DRIVE_RESULT_ROOT,
        "--config-dir", BASELINE_CONFIG_DIR,
        "--external-root", EXTERNAL_BASELINES_ROOT,
        "--gpu-profile-interval-seconds", str(BASELINE_GPU_PROFILE_INTERVAL_SECONDS),
    ]
    if VIDEOSEAL_INPUT_VIDEO_PATH:
        videoseal_args.extend(["--input-video-path", VIDEOSEAL_INPUT_VIDEO_PATH])
    run_python_script("scripts/prepare_baselines/run_videoseal_real_smoke.py", *videoseal_args)
else:
    print("已跳过 external_videoseal 真实 smoke。")


## 9. external_rivagan 真实可运行性 smoke

该步骤会真实加载 RivaGAN、下载 32-bit 权重、计算权重 digest、执行单视频 encode / decode, 并额外执行 H.264 CRF 28 后的 decode。该结果仍然只是单 baseline smoke, 不能替代正式 fixed-FPR baseline comparison。


In [ ]:
if RUN_RIVAGAN_REAL_SMOKE:
    rivagan_args = [
        "--run-root", "/content/TSTW_runtime/runs/external_rivagan_real_smoke",
        "--result-root", DRIVE_RESULT_ROOT,
        "--config-dir", BASELINE_CONFIG_DIR,
        "--external-root", EXTERNAL_BASELINES_ROOT,
        "--gpu-profile-interval-seconds", str(BASELINE_GPU_PROFILE_INTERVAL_SECONDS),
    ]
    if RIVAGAN_INPUT_VIDEO_PATH:
        rivagan_args.extend(["--input-video-path", RIVAGAN_INPUT_VIDEO_PATH])
    run_python_script("scripts/prepare_baselines/run_rivagan_real_smoke.py", *rivagan_args)
else:
    print("已跳过 external_rivagan 真实 smoke。")


## 10. external_hidden_framewise 真实可运行性 smoke

该步骤会真实加载 HiDDeN 图像水印模型, 使用上游 combined-noise checkpoint, 对视频帧逐帧执行 encode / decode, 并额外执行 H.264 CRF 28 后的 decode。该结果仍然只是图像水印逐帧迁移 baseline smoke, 不能替代正式 fixed-FPR baseline comparison。


In [ ]:
if RUN_HIDDEN_FRAMEWISE_REAL_SMOKE:
    hidden_args = [
        "--run-root", "/content/TSTW_runtime/runs/external_hidden_framewise_real_smoke",
        "--result-root", DRIVE_RESULT_ROOT,
        "--config-dir", BASELINE_CONFIG_DIR,
        "--external-root", EXTERNAL_BASELINES_ROOT,
        "--gpu-profile-interval-seconds", str(BASELINE_GPU_PROFILE_INTERVAL_SECONDS),
    ]
    if HIDDEN_FRAMEWISE_INPUT_VIDEO_PATH:
        hidden_args.extend(["--input-video-path", HIDDEN_FRAMEWISE_INPUT_VIDEO_PATH])
    run_python_script("scripts/prepare_baselines/run_hidden_framewise_real_smoke.py", *hidden_args)
else:
    print("已跳过 external_hidden_framewise 真实 smoke。")


## 11. 汇总三个真实 smoke 结果包

该步骤读取 Drive 中最新的 `external_videoseal`、`external_rivagan` 和 `external_hidden_framewise` 真实 smoke 结果包, 生成 JSON、CSV 和 Markdown 摘要。该摘要只用于进入正式 baseline comparison 前的工程检查, 不支持论文 claim。


In [ ]:
if RUN_BASELINE_REAL_SMOKE_SUMMARY:
    run_python_script(
        "scripts/prepare_baselines/summarize_baseline_real_smoke.py",
        "--result-root", DRIVE_RESULT_ROOT,
        "--output-dir", BASELINE_REAL_SMOKE_SUMMARY_DIR,
    )
else:
    print("已跳过 baseline real smoke 汇总。")


## 12. 准备正式 baseline comparison runner 输入契约

该步骤读取已经解压的阶段二 records 和真实 smoke 汇总, 冻结正式 baseline comparison runner 的输入宇宙, 包括 split、sample role、attack name、内部方法变体和 target FPR。它仍然不生成论文主表, 只生成正式 runner 的输入契约。


In [ ]:
real_smoke_summary_json = f"{BASELINE_REAL_SMOKE_SUMMARY_DIR}/baseline_real_smoke_summary.json"
formal_input_short_commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=REPO_DIR,
    text=True,
).strip()
formal_input_args = [
    "--stage-two-package-root", EXTRACTED_REAL_VIDEO_VAE_PACKAGE_ROOT,
    "--run-root", BASELINE_FORMAL_RUN_ROOT,
    "--result-root", DRIVE_RESULT_ROOT,
    "--short-commit", formal_input_short_commit,
]
for baseline_name in BASELINE_FORMAL_SCORING_EXECUTION_BASELINE_NAMES:
    formal_input_args.extend(["--baseline-name", baseline_name])
if BASELINE_FORMAL_SCORING_OVERWRITE_DRIVE_RESULT:
    formal_input_args.append("--overwrite")
if Path(real_smoke_summary_json).exists():
    formal_input_args.extend(["--real-smoke-summary", real_smoke_summary_json])
else:
    print(
        "未找到 baseline real smoke summary, 将省略可选 --real-smoke-summary 参数: "
        + real_smoke_summary_json
    )
run_python_script(
    "scripts/prepare_baselines/prepare_baseline_comparison_formal_inputs.py",
    *formal_input_args,
)


## 13. 生成正式 baseline comparison scoring work-item 计划

该步骤读取 formal input contract 和阶段二 records, 生成外部 baseline 的正式 scoring work items, 并在成功后复制到 Drive。该步骤仍不伪造分数, 也不生成 TPR/FPR 表。后续执行层必须基于这些 work items 完成 embed、attack、detect、calibration 和 test 冻结。


In [ ]:
if RUN_BASELINE_FORMAL_SCORING_PLAN:
    formal_input_contract_path = f"{BASELINE_FORMAL_RUN_ROOT}/configs/baseline_comparison_formal_input_contract.json"
    scoring_args = [
        "--run-root", BASELINE_FORMAL_SCORING_PLAN_RUN_ROOT,
        "--stage-two-package-root", EXTRACTED_REAL_VIDEO_VAE_PACKAGE_ROOT,
        "--formal-input-contract", formal_input_contract_path,
        "--config-dir", BASELINE_CONFIG_DIR,
        "--external-root", EXTERNAL_BASELINES_ROOT,
        "--shard-count", str(BASELINE_SCORING_SHARD_COUNT),
        "--shard-index", str(BASELINE_SCORING_SHARD_INDEX),
        "--result-root", DRIVE_RESULT_ROOT,
        "--short-commit", formal_input_short_commit,
    ]
    if BASELINE_FORMAL_SCORING_OVERWRITE_DRIVE_RESULT:
        scoring_args.append("--overwrite")
    for baseline_name in BASELINE_SCORING_BASELINE_NAMES:
        scoring_args.extend(["--baseline-name", baseline_name])
    run_python_script("scripts/prepare_baselines/run_baseline_comparison_formal_scoring.py", *scoring_args)
else:
    print("已跳过 baseline comparison formal scoring work-item 计划。")


In [ ]:
if RUN_BASELINE_FORMAL_SCORING_SMALL_VALIDATION:
    formal_input_contract_path = f"{BASELINE_FORMAL_RUN_ROOT}/configs/baseline_comparison_formal_input_contract.json"
    stage_two_artifact_roots = resolve_stage_two_artifact_roots_for_baseline()
    execution_args = [
        "--execute",
        "--run-root", BASELINE_FORMAL_SCORING_EXECUTION_RUN_ROOT,
        "--stage-two-package-root", EXTRACTED_REAL_VIDEO_VAE_PACKAGE_ROOT,
        "--formal-input-contract", formal_input_contract_path,
        "--config-dir", BASELINE_CONFIG_DIR,
        "--external-root", EXTERNAL_BASELINES_ROOT,
        "--shard-count", str(BASELINE_SCORING_SHARD_COUNT),
        "--shard-index", str(BASELINE_SCORING_SHARD_INDEX),
        "--worker-count", str(BASELINE_FORMAL_SCORING_WORKER_COUNT),
        "--batch-size", str(BASELINE_FORMAL_SCORING_BATCH_SIZE),
        "--gpu-profile-interval-seconds", str(BASELINE_GPU_PROFILE_INTERVAL_SECONDS),
        "--result-root", DRIVE_RESULT_ROOT,
        "--short-commit", formal_input_short_commit,
    ]
    for artifact_root in stage_two_artifact_roots:
        execution_args.extend(["--stage-two-artifact-root", artifact_root])
    if BASELINE_FORMAL_SCORING_OVERWRITE_DRIVE_RESULT:
        execution_args.append("--overwrite")
    if BASELINE_FORMAL_SCORING_EXECUTION_MAX_WORK_ITEMS is not None:
        execution_args.extend(["--max-work-items", str(BASELINE_FORMAL_SCORING_EXECUTION_MAX_WORK_ITEMS)])
    for baseline_name in BASELINE_FORMAL_SCORING_EXECUTION_BASELINE_NAMES:
        execution_args.extend(["--baseline-name", baseline_name])
    run_python_script("scripts/prepare_baselines/run_baseline_comparison_formal_scoring.py", *execution_args)
else:
    print("已跳过 baseline comparison formal scoring 小规模执行验证。")


## 14. 当前结论

如果上面的所有单元执行成功, 说明阶段二 real-video VAE 正式结果包可复用, 且阶段三三个外部 baseline 的 Colab 冷启动真实 smoke 链路已经具备工程可运行性。

- `external_videoseal` 和 `external_rivagan` 若在汇总中为 `real_smoke_passed`, 只能说明单视频 clean/H.264 smoke 可运行。
- `external_hidden_framewise` 若在汇总中为 `real_smoke_executed_negative`, 表示该图像水印逐帧迁移 baseline 可运行但低于 smoke 阈值, 应进入 limitation, 不能被写成原生视频水印成功 baseline。
- 下一步仍需实现并运行正式 fixed-FPR baseline comparison 评分 runner: 从本 Notebook 解压出的阶段二包读取同一 split、同一 attack matrix、calibration split 阈值依据、test split records、表格和 claim audit 所需输入。


## 9. 当前 baseline shard 完成性检查

该步骤只检查 `shard_runs` 是否完成。score records 聚合必须使用 `aggregate_baseline_comparison_gate_shards.ipynb`。


In [ ]:
from pathlib import Path


def _baseline_result_label(baseline_names):
    normalized = sorted({name.strip() for name in baseline_names if name and name.strip()})
    if len(normalized) == 1:
        return normalized[0]
    return "multi_baseline"

baseline_result_label = _baseline_result_label(BASELINE_FORMAL_SCORING_EXECUTION_BASELINE_NAMES)

scoring_plan_label = _baseline_result_label(BASELINE_SCORING_BASELINE_NAMES)
expected_plan_run_id = f"baseline_comparison_formal_scoring_plan_{scoring_plan_label}_sc{BASELINE_SCORING_SHARD_COUNT:02d}_si{BASELINE_SCORING_SHARD_INDEX:02d}_{formal_input_short_commit[:7]}"
expected_plan_root = Path(DRIVE_RESULT_ROOT) / "baseline_comparison_gate" / scoring_plan_label / "scoring_plans" / expected_plan_run_id
expected_formal_input_run_id = f"baseline_comparison_formal_inputs_{baseline_result_label}_{formal_input_short_commit[:7]}"
# formal input 实际 run_id 包含 UTC 时间, 因此 completion summary 仅报告所在目录。
expected_formal_input_root = Path(DRIVE_RESULT_ROOT) / "baseline_comparison_gate" / baseline_result_label / "formal_inputs"
expected_run_id = f"baseline_comparison_formal_scoring_execution_{baseline_result_label}_sc{BASELINE_SCORING_SHARD_COUNT:02d}_si{BASELINE_SCORING_SHARD_INDEX:02d}_{formal_input_short_commit[:7]}"
expected_drive_root = Path(DRIVE_RESULT_ROOT) / "baseline_comparison_gate" / baseline_result_label / "shard_runs" / expected_run_id
required_paths = {
    "score_records": expected_drive_root / "records" / "baseline_formal_score_records.jsonl",
    "execution_manifest": expected_drive_root / "artifacts" / "baseline_comparison_formal_scoring_execution_manifest.json",
    "formal_input_contract_snapshot": expected_drive_root / "configs" / "baseline_comparison_formal_input_contract.json",
    "gpu_trace": expected_drive_root / "runtime_profile" / "baseline_gpu_profiles" / "formal_scoring_execution" / "gpu_runtime_trace.csv",
    "gpu_summary": expected_drive_root / "runtime_profile" / "baseline_gpu_profiles" / "formal_scoring_execution" / "gpu_runtime_summary.json",
}
completion_summary = {
    "result_flow": "baseline_shard_run",
    "baseline": baseline_result_label,
    "shard_count": BASELINE_SCORING_SHARD_COUNT,
    "shard_index": BASELINE_SCORING_SHARD_INDEX,
    "drive_formal_input_root": str(expected_formal_input_root),
    "drive_scoring_plan_root": str(expected_plan_root),
    "drive_shard_run_root": str(expected_drive_root),
    "required_paths": {name: path.exists() for name, path in required_paths.items()},
    "next_notebook": "aggregate_baseline_comparison_gate_shards.ipynb",
}
completion_summary["status"] = all(completion_summary["required_paths"].values())
print(completion_summary)
if RUN_BASELINE_FORMAL_SCORING_SMALL_VALIDATION and not completion_summary["status"]:
    raise RuntimeError(completion_summary)
